# FacetLens - a walkthrough

Scores conversation text against a 399-facet catalogue, and decides which facets
may legitimately be scored at all.

**Generated by `python -m src.pipeline notebook`.** Every output below was
produced by executing the cell above it - nothing is typed by hand. Every cell
uses `MockBackend`, so this reproduces with no model and no network.

The hard part of this assignment is not prompting an LLM for scores. It is
refusing to produce scores the conversation does not support.

## 1. The problem, in one cell

The catalogue mixes things a conversation can evidence with things it cannot.

In [ ]:
import csv, collections
rows = list(csv.DictReader(open("data/processed/enriched_facets.csv", encoding="utf-8")))
observable = sum(1 for r in rows if r["conversation_observable"] == "True")
print(f"{len(rows)} facets, {observable} conversation-observable\n")
for t, n in collections.Counter(r["facet_type"] for r in rows).most_common(7):
    row = next(r for r in rows if r["facet_type"] == t)
    tag = "scorable" if row["conversation_observable"] == "True" else "NOT scorable"
    print(f"  {n:3d}  {t:36s} {tag:12s} e.g. {row['facet_raw'][:34]}")

399 facets, 235 conversation-observable

  100  personality_trait                    scorable     e.g. Naivety
   53  quantified_activity_metric           NOT scorable e.g. Pilgrimage participation count
   31  instrument_or_scale_header           NOT scorable e.g. Democratic Leadership:
   28  cognitive_test_ability               NOT scorable e.g. Statistical Reasoning
   27  interpersonal                        scorable     e.g. Assertiveness and control in relat
   22  emotional_state                      scorable     e.g. Discontentment
   20  behavioral_tendency                  scorable     e.g. Risktaking


## 2. Four gates

A retriever is perfectly happy to return `FSH level` for "I've been so tired
lately". Knowing that *retrieving* a facet and being able to *score* it are
different questions is the whole design.

In [ ]:
from src.retrieval.embed import build_index
from src.retrieval.retrieve import route

index = build_index()
text = "Honestly I have been so tired the last few weeks, dragging myself out of bed."
routed = route(text, index, top_k=12)
total = len(routed.scorable) + len(routed.gated_out) + len(routed.policy_blocked)

print("CONVERSATION:", text, "\n")
print(f"GATE 1  retrieved {total} candidates by similarity alone\n")
print(f"GATE 2  blocked {len(routed.gated_out)} as not observable - no LLM call, no tokens:")
for c in routed.gated_out[:5]:
    print(f"          {c.facet[:32]:34s} {c.abstention_reason}")
print(f"\nGATE 3  sends {len(routed.scorable)} facets to the model:")
for c in routed.scorable[:5]:
    print(f"          {c.facet[:32]:34s} {c.facet_type}")

CONVERSATION: Honestly I have been so tired the last few weeks, dragging myself out of bed. 

GATE 1  retrieved 12 candidates by similarity alone

GATE 2  blocked 2 as not observable - no LLM call, no tokens:
          Compassion Fatigue                 requires_clinical_assessment
          Burnout Symptoms                   requires_clinical_assessment

GATE 3  sends 10 facets to the model:
          Discontentment                     emotional_state
          Moroseness                         emotional_state
          Boredom Susceptibility             emotional_state
          General Mood and Attitude          emotional_state
          Vivacity                           emotional_state


## 3. The hallucination challenge

Three conversations where a naive scorer would confidently infer something the
text does not justify. The medical, financial and religious facets are refused
**before** the model sees them, so the guarantee does not depend on a 7B model
choosing to decline.

In [ ]:
import json
RISKY = {"biometric_physiological", "clinical_mental_health",
         "quantified_activity_metric", "demographic_biographical",
         "spiritual_religious_practice_metric", "cognitive_test_ability"}
for line in open("artifacts/benchmark_results.jsonl", encoding="utf-8"):
    r = json.loads(line)
    if not r["conversation_id"].startswith("h"):
        continue
    scored = [v for v in r["verdicts"] if v["status"] == "scored"]
    refused = [v for v in r["verdicts"] if v["status"] in ("not_observable", "policy_blocked")]
    leaked = [v for v in scored if v["facet_type"] in RISKY]
    print(f'{r["conversation_id"]}  "{r["conversation"][:58]}..."')
    print(f'   {len(scored)} scored, {len(refused)} refused   RISKY FACETS SCORED: {len(leaked)}')
    for v in refused[:2]:
        print(f'      refused {v["facet"][:30]:32s} {v["reason"][:48]}')
    print()

h01_tired  "Honestly I've been so tired the last few weeks. Dragging m..."
   3 scored, 15 refused   RISKY FACETS SCORED: 0
      refused Compassion Fatigue               Not scorable from conversation: classified as cl
      refused Breakfast-skipping frequency     Not scorable from conversation: classified as qu

h02_saving  "I cut my spending a lot this year and I'm finally managing..."
   2 scored, 7 refused   RISKY FACETS SCORED: 0
      refused Observing                        Refused by default policy: this facet concerns r
      refused Quran khatam cycles per year     Not scorable from conversation: classified as qu

h03_practice  "I feel so calm and centred after my morning practice. It's..."
   8 scored, 8 refused   RISKY FACETS SCORED: 0
      refused Mantra meditation                Not scorable from conversation: classified as qu
      refused Types of Mindfulness Technique   Refused by default policy: this facet concerns r



## 4. Robustness: the model misbehaving on purpose

`MockBackend` emits deliberately broken responses. A malformed batch must never
take down a run, and a fabricated evidence quote must never survive as a score.

In [ ]:
import collections
from src.scoring.scorer import score_conversation
from src.scoring.backends import MockBackend

text = "I led a team of five engineers and assigned tasks based on their strengths."
for mode in ["valid", "malformed", "partial", "bad_schema", "fabricated"]:
    result = score_conversation(text, top_k=8, batch_size=4,
                                backend=MockBackend(mode=mode), index=index)
    counts = collections.Counter(v.status.value for v in result.verdicts)
    print(f"  backend={mode:11s} -> {dict(counts)}")
print()
print("No mode crashes the pipeline. 'fabricated' is caught by the evidence")
print("verifier and downgraded to insufficient_evidence rather than scored.")

  backend=valid       -> {'scored': 8}
  backend=malformed   -> {'error': 8}
  backend=partial     -> {'scored': 2, 'error': 6}
  backend=bad_schema  -> {'error': 8}
  backend=fabricated  -> {'insufficient_evidence': 8}

No mode crashes the pipeline. 'fabricated' is caught by the evidence
verifier and downgraded to insufficient_evidence rather than scored.


## 5. Results

Full detail in [`artifacts/benchmark_report.md`](benchmark_report.md).

In [ ]:
import re
KEYS = ("Status agreement", "Correct abstentions", "Within", "Exact score",
        "Missed abstentions", "False abstentions", "ended as")
for line in open("artifacts/benchmark_report.md", encoding="utf-8"):
    if line.startswith("- ") and any(k in line for k in KEYS):
        print("  " + re.sub(r"\*\*", "", line[2:]).rstrip())

  Status agreement (scored vs abstained vs not_observable): 48/55 (87.3%)
  Exact score agreement (both scored): 3/15 (20.0%)
  Within +/-1: 13/15 (86.7%)
  Correct abstentions: 33/36 (91.7%)
  Missed abstentions (system scored something the reference says is unsupported): 3
  False abstentions (system abstained where the reference expects a score): 4
  Verdicts that ended as `error`: 1


### Where it is weak, stated plainly

- **Retrieval recall 34.5%** is the softest number. Five interventions were
  measured and four failed. The fifth - a cross-encoder reranker - *improved
  retrieval* to 40.0% and *worsened* end-to-end agreement, so it ships as an
  option rather than the default (DECISIONS.md D12).
- **Exact score agreement 20%** is a scale-calibration artefact, not scoring
  failure: 8 of 15 scored pairs are off by exactly +1 because the reference
  labels encode an earlier anchor definition. The labels were deliberately
  **not** recalibrated after seeing that result.
- **The taxonomy has a measured ~10% error rate**, from a seeded 30-row manual
  review that found 4 misclassifications - all in the dangerous direction
  (DEBUGGING.md #7).
- **Self-reported confidence is uninformative**: the 0.9 bucket agrees 44% of
  the time. Reported rather than presented as a quality signal.

`DEBUGGING.md` records 12 real failures, three of them my own reasoning errors
caught by measurement rather than review.